In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, gzip, tarfile, shutil, subprocess, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [10]:
# =============================================================================
# Cell 2 - UGR'16 config and where the raw week files go.
# We use the TEST set (July week5 + August weeks): the labeled attacks live
# there. Ladder = calibrate on July, evaluate successive August weeks (the
# documented July->August drift). Start with july_week5 + one August week.
# =============================================================================
RAW_DIR = Path('/content/ugr16_raw'); RAW_DIR.mkdir(parents=True, exist_ok=True)   # local disk (big files)
OUT_DIR = config.DATASETS_DIR / 'ugr16'; OUT_DIR.mkdir(parents=True, exist_ok=True) # Drive (small parquet)

# UGR'16 labeled CSV columns (no header in the source files)
UGR_COLS = ['te','td','sa','da','sp','dp','pr','flg','fwd','stos','pkt','byt','label']

# background label; everything else is an attack/anomaly class
BACKGROUND = 'background'

# per-week subsample caps (keep the study tractable; attacks kept up to the cap)
BG_CAP_PER_WEEK      = 200_000
ATTACK_CAP_PER_CLASS = 50_000

# expected week tags -> the source CSV file name stem on nesg (place the file with the tag in its name)
WEEKS = ['july_week5','august_week1','august_week2','august_week3','august_week4','august_week5']

found = sorted([p for p in RAW_DIR.glob('*') if p.is_file()])
print('raw files on local disk:', len(found))
for p in found: print('  ', p.name, f'{p.stat().st_size/1e9:.2f} GB')


raw files on local disk: 2
   august_week5.csv.tar.gz 0.57 GB
   july_week5.csv.tar.gz 7.71 GB


In [8]:
!mkdir -p /content/ugr16_raw
!wget -c -O /content/ugr16_raw/july_week5.csv.tar.gz "https://nesg.ugr.es/nesg-ugr16/download/attack/july/week5/july_week5_csv.tar.gz"

--2026-07-25 18:47:09--  https://nesg.ugr.es/nesg-ugr16/download/attack/july/week5/july_week5_csv.tar.gz
Resolving nesg.ugr.es (nesg.ugr.es)... 150.214.190.73
Connecting to nesg.ugr.es (nesg.ugr.es)|150.214.190.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7709006531 (7.2G) [application/x-gzip]
Saving to: ‘/content/ugr16_raw/july_week5.csv.tar.gz’

/content/ugr16_raw/ 100%[===================>]   7.18G  8.49MB/s    in 14m 31s 

2026-07-25 19:01:40 (8.44 MB/s) - ‘/content/ugr16_raw/july_week5.csv.tar.gz’ saved [7709006531/7709006531]



In [9]:
!wget -c -O /content/ugr16_raw/august_week5.csv.tar.gz "https://nesg.ugr.es/nesg-ugr16/download/attack/august/week5/august_week5_csv.tar.gz"

--2026-07-25 19:04:05--  https://nesg.ugr.es/nesg-ugr16/download/attack/august/week5/august_week5_csv.tar.gz
Resolving nesg.ugr.es (nesg.ugr.es)... 150.214.190.73
Connecting to nesg.ugr.es (nesg.ugr.es)|150.214.190.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 571031116 (545M) [application/x-gzip]
Saving to: ‘/content/ugr16_raw/august_week5.csv.tar.gz’

/content/ugr16_raw/ 100%[===================>] 544.58M  8.76MB/s    in 63s     

2026-07-25 19:05:08 (8.65 MB/s) - ‘/content/ugr16_raw/august_week5.csv.tar.gz’ saved [571031116/571031116]



In [11]:
# =============================================================================
# Cell 3 - if no raw files present, print exact acquisition steps and stop.
# =============================================================================
if not found:
    print('NO UGR-16 WEEK FILES on the Colab local disk yet.')
    print()
    print('Get them from the nesg site (they are ~14GB each, so put them on the')
    print('LOCAL disk /content/ugr16_raw, NOT Drive):')
    print('  1. Open https://nesg.ugr.es/nesg-ugr16/download/attack/july/week5/july_week5_csv.tar.gz  ->  TEST  ->  July  ->  Week #5')
    print('  2. Download the file named "july_week5_csv" (Collected CSV netflow flows, labeled).')
    print('  3. Repeat for August -> at least one week (e.g. august week1) to get a drift rung.')
    print('  4. Put the downloaded files in /content/ugr16_raw/ and make sure each filename')
    print('     contains its week tag, one of:', WEEKS)
    print()
    print('If you can copy the download link from the browser, you can fetch straight to Colab:')
    print('  !wget -O /content/ugr16_raw/july_week5.csv.tar "https://nesg.ugr.es/nesg-ugr16/download/attack/july/week5/july_week5_csv.tar.gz"')
    print()
    print('Start with just july_week5 + one August week; that is enough to prove the pipeline.')
    raise SystemExit('place week files in /content/ugr16_raw, then re-run')


In [13]:
import tarfile
from pathlib import Path

for p in sorted(Path('/content/ugr16_raw').glob('*.tar.gz')):
    print(f'\n=== {p.name}  ({p.stat().st_size/1e9:.2f} GB) ===')
    with tarfile.open(p, 'r:*') as tf:
        for m in tf.getmembers():
            print(f'  {m.name}   {m.size/1e9:.2f} GB   {"dir" if m.isdir() else "file"}')


=== august_week5.csv.tar.gz  (0.57 GB) ===
  uniq/august.week5.csv.uniqblacklistremoved   3.89 GB   file

=== july_week5.csv.tar.gz  (7.71 GB) ===
  uniq/july.week5.csv.uniqblacklistremoved   52.10 GB   file


In [14]:
# =============================================================================
# Cell 4 - PEEK: verify the schema before trusting it (the file may be a tar, a
# gzip, or plain csv; header present or not). Prints the first rows and the
# detected format for each raw file.
# =============================================================================
def open_text(path):
    # returns (text handle, holder-to-close). Picks the CSV by largest member,
    # since UGR'16 tars name the file without a .csv extension.
    import io
    name = path.name.lower()
    if name.endswith(('.tar', '.tar.gz', '.tgz')):
        tf = tarfile.open(path, 'r:*')
        files = [m for m in tf.getmembers() if m.isfile()]
        assert files, f'no files inside {path.name}'
        member = max(files, key=lambda m: m.size)   # the CSV is the big one
        print(f'  [{path.name}] reading member: {member.name} ({member.size/1e9:.1f} GB)')
        return io.TextIOWrapper(tf.extractfile(member), encoding='utf-8', errors='replace'), tf
    if name.endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8', errors='replace'), None
    return open(path, 'rt', encoding='utf-8', errors='replace'), None
for p in found:
    h, holder = open_text(p)
    first = [next(h) for _ in range(3)]
    h.close();
    if holder is not None: holder.close()
    ncols = first[0].count(',') + 1
    print(f'{p.name}: {ncols} columns, first row:')
    print('   ', first[0].strip()[:160])
    print('   header?', not first[0].split(',')[0].replace(':','').replace('-','').replace(' ','').isdigit())


  [august_week5.csv.tar.gz] reading member: uniq/august.week5.csv.uniqblacklistremoved (3.9 GB)
august_week5.csv.tar.gz: 1 columns, first row:
    
   header? True
  [july_week5.csv.tar.gz] reading member: uniq/july.week5.csv.uniqblacklistremoved (52.1 GB)
july_week5.csv.tar.gz: 1 columns, first row:
    
   header? True


In [15]:
# =============================================================================
# Cell 5 - PROCESS each raw week file: stream in chunks, keep all attacks (up to
# the cap) and a capped background sample, preserve the flow-end timestamp, tag
# the week, write a compact parquet to Drive. Resumable: skips weeks already done.
# Adjust HAS_HEADER if cell 4 showed a header row.
# =============================================================================
HAS_HEADER = False           # set True if cell 4 reported a header
CHUNK = 2_000_000

def week_of(fname):
    low = fname.lower()
    return next((w for w in WEEKS if w in low), None)

for p in found:
    wk = week_of(p.name)
    if wk is None:
        print('SKIP (no week tag in name):', p.name); continue
    out = OUT_DIR / f'{wk}.parquet'
    if out.exists():
        print('already processed:', wk); continue

    t0 = time.time(); rng = np.random.default_rng(20260725)
    kept_bg = []; kept_att = {}; bg_seen = 0
    h, holder = open_text(p)
    reader = pd.read_csv(h, header=0 if HAS_HEADER else None, names=None if HAS_HEADER else UGR_COLS,
                         chunksize=CHUNK, low_memory=False)
    for ci, chunk in enumerate(reader):
        chunk.columns = [str(c).strip() for c in chunk.columns]
        lab = chunk['label'].astype(str).str.strip().str.lower()
        att = chunk[lab != BACKGROUND]
        bg  = chunk[lab == BACKGROUND]
        for cls, sub in att.groupby(lab[lab != BACKGROUND]):
            cur = kept_att.get(cls)
            need = ATTACK_CAP_PER_CLASS - (0 if cur is None else len(cur))
            if need > 0:
                take = sub if len(sub) <= need else sub.sample(need, random_state=int(rng.integers(1<<30)))
                kept_att[cls] = take if cur is None else pd.concat([cur, take])
        # reservoir-style background sampling toward the cap
        bg_seen += len(bg)
        if bg_seen <= BG_CAP_PER_WEEK:
            kept_bg.append(bg)
        else:
            frac = max(0.0, (BG_CAP_PER_WEEK*1.5 - sum(len(x) for x in kept_bg)) / max(len(bg),1))
            if frac > 0: kept_bg.append(bg.sample(frac=min(frac,1.0), random_state=int(rng.integers(1<<30))))
        print(f'  {wk} chunk {ci}: +{len(att)} att, bg_seen={bg_seen:,}  [{(time.time()-t0)/60:.1f} min]')
    h.close()
    if holder is not None: holder.close()

    bg_all = pd.concat(kept_bg) if kept_bg else pd.DataFrame(columns=UGR_COLS)
    if len(bg_all) > BG_CAP_PER_WEEK:
        bg_all = bg_all.sample(BG_CAP_PER_WEEK, random_state=7)
    parts = [bg_all] + list(kept_att.values())
    wkdf = pd.concat(parts, ignore_index=True)
    wkdf['week'] = wk
    wkdf['te'] = pd.to_datetime(wkdf['te'], errors='coerce')
    wkdf.to_parquet(out, index=False)
    print(f'{wk}: wrote {len(wkdf):,} rows -> {out.name}')
    print('   class counts:', wkdf['label'].astype(str).str.strip().str.lower().value_counts().to_dict())


  [august_week5.csv.tar.gz] reading member: uniq/august.week5.csv.uniqblacklistremoved (3.9 GB)
  august_week5 chunk 0: +8940 att, bg_seen=1,991,060  [0.4 min]
  august_week5 chunk 1: +6778 att, bg_seen=3,984,282  [0.5 min]
  august_week5 chunk 2: +6016 att, bg_seen=5,978,266  [0.6 min]
  august_week5 chunk 3: +6469 att, bg_seen=7,971,797  [0.8 min]
  august_week5 chunk 4: +6143 att, bg_seen=9,965,654  [0.9 min]
  august_week5 chunk 5: +6723 att, bg_seen=11,958,931  [1.0 min]
  august_week5 chunk 6: +6302 att, bg_seen=13,952,629  [1.1 min]
  august_week5 chunk 7: +8945 att, bg_seen=15,943,684  [1.2 min]
  august_week5 chunk 8: +7401 att, bg_seen=17,936,283  [1.3 min]
  august_week5 chunk 9: +7069 att, bg_seen=19,929,214  [1.5 min]
  august_week5 chunk 10: +7815 att, bg_seen=21,921,399  [1.6 min]
  august_week5 chunk 11: +6730 att, bg_seen=23,914,669  [1.7 min]
  august_week5 chunk 12: +6210 att, bg_seen=25,908,459  [1.8 min]
  august_week5 chunk 13: +6964 att, bg_seen=27,901,495  [1.9 

In [16]:
# =============================================================================
# Cell 6 - inventory across weeks + commit (parquet stays in data/, gitignored).
# =============================================================================
weeks_done = sorted(OUT_DIR.glob('*.parquet'))
inv = []
for p in weeks_done:
    df = pd.read_parquet(p, columns=['label','week','te'])
    lab = df['label'].astype(str).str.strip().str.lower()
    row = {'week': p.stem, 'rows': len(df),
           'te_min': str(df['te'].min()), 'te_max': str(df['te'].max())}
    row.update(lab.value_counts().to_dict())
    inv.append(row)
if inv:
    idf = pd.DataFrame(inv).fillna(0)
    idf.to_csv(config.REPORTS_DIR/'ugr16_week_inventory.csv', index=False)
    print(idf.to_string(index=False))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb14: UGR16 test-set acquisition + per-week subsample (July->August)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


        week   rows              te_min              te_max  background  blacklist  anomaly-sshscan  anomaly-spam     dos  scan11  scan44  nerisbotnet
august_week5 250001 2016-08-29 00:12:21 2016-08-29 03:48:21      200000      50000                1           0.0     0.0     0.0     0.0          0.0
  july_week5 500010 2016-07-27 13:43:28 2016-07-31 20:31:07      200000      50000               10       50000.0 50000.0 50000.0 50000.0      50000.0
[main 28ad43b] nb14: UGR16 test-set acquisition + per-week subsample (July->August)
 3 files changed, 5 insertions(+), 1 deletion(-)
 rewrite notebooks/13_cic_conformal_coverage.ipynb (89%)
 create mode 100644 notebooks/14_ugr16_acquisition.ipynb
 create mode 100644 reports/ugr16_week_inventory.csv
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   917f518..28ad43b  main -> main
28ad43b nb14: UGR16 test-set acquisition + per-week subsample (July->August)
917f518 nb13: 